# Fine-tuning Llama-3 para Assistente Médico (Fase 3) - Dados Reais

Este notebook realiza o fine-tuning de um modelo Llama-3 usando datasets reais de Saúde da Mulher (MedQuAD e Women Health).

In [1]:
!pip install -q -U transformers peft bitsandbytes datasets accelerate trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 69.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 751.0/751.0 kB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 9.5 MB/s eta 0:00:00


In [2]:
from datasets import load_dataset
import json
import os
import shutil

def prepare_real_data():
    # Se subir o arquivo gerado localmente ou na pasta data, da pra usar ele
    if os.path.exists("real_medical_data.jsonl"):
        print("Dataset real já existe localmente.")
        return
    elif os.path.exists("../data/real_medical_data.jsonl"):
        print("Copiando dataset real da pasta data...")
        shutil.copy("../data/real_medical_data.jsonl", "real_medical_data.jsonl")
        return

    print("Baixando datasets reais do Hugging Face (MedQuAD e Women Health)...")
    processed_data = []

    # 1. Carregar Women Health
    try:
        ds_women = load_dataset("altaidevorg/women-health-mini", split="train")
        for item in ds_women:
            conversations = item.get("conversations", [])
            instruction = ""
            response = ""
            for msg in conversations:
                if msg.get("role") == "user":
                    instruction = msg.get("content", "")
                elif msg.get("role") == "assistant":
                    response = msg.get("content", "")
            if instruction and response:
                processed_data.append({
                    "instruction": instruction,
                    "context": "Saúde da Mulher / Ginecologia",
                    "response": response
                })
        print(f"Women Health carregado: {len(processed_data)} exemplos.")
    except Exception as e:
        print(f"Erro ao carregar Women Health: {e}")

    # 2. Carregar MedQuAD (Subset filtrado)
    try:
        ds_medquad = load_dataset("keivalya/MedQuad-MedicalQnADataset", split="train")
        keywords = ["woman", "women", "pregnancy", "breast", "gynecology", "obstetrics", "menstrual", "urology"]
        ds_medquad_filtered = ds_medquad.filter(lambda x: any(k in x['Question'].lower() for k in keywords))
        for item in ds_medquad_filtered:
            processed_data.append({
                "instruction": item.get("Question", ""),
                "context": "Saúde da Mulher / Triagem Médica",
                "response": item.get("Answer", "")
            })
        print(f"MedQuAD filtrado adicionado. Total: {len(processed_data)} exemplos.")
    except Exception as e:
        print(f"Erro ao carregar MedQuAD: {e}")

    if processed_data:
        with open("real_medical_data.jsonl", "w", encoding="utf-8") as f:
            for entry in processed_data:
                f.write(json.dumps(entry, ensure_ascii=False) + "\n")
        print(f"Dataset real preparado com sucesso: real_medical_data.jsonl ({len(processed_data)} exemplos)")
    else:
        print("Nenhum dado pôde ser preparado.")

prepare_real_data()

Baixando datasets reais do Hugging Face (MedQuAD e Women Health)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


women-health-mini.jsonl:   0%|          | 0.00/35.9M [00:00<?, ?B/s]

...

README.md:   0%|          | 0.00/233 [00:00<?, ?B/s]

medDataset_processed.csv:   0%|          | 0.00/22.5M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/16407 [00:00<?, ? examples/s]

Filter:   0%|          | 0/16407 [00:00<?, ? examples/s]

MedQuAD filtrado adicionado. Total: 10468 exemplos.
Dataset real preparado com sucesso: real_medical_data.jsonl (10468 exemplos)


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

model_id = "unsloth/llama-3-8b-bnb-4bit"
dataset_path = "real_medical_data.jsonl"

dataset = load_dataset("json", data_files=dataset_path, split="train")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

training_args = SFTConfig(
    output_dir="./llama3-medical-fase3",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    max_steps=100,
    logging_steps=10,
    fp16=False,
    bf16=False,
    dataset_text_field="instruction",
    max_length=512,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    processing_class=tokenizer,
    args=training_args
)

trainer.train()
print("Treinamento Concluído!")

Generating train split: 0 examples [00:00, ? examples/s]

config.json: 0.00B [00:00, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
/usr/local/lib/python3.12/dist-packages/transformers/quantizers/auto.py:262: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

...

special_tokens_map.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

Adding EOS to train dataset:   0%|          | 0/10468 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/10468 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 128001}.


Step,Training Loss
10,2.665528
20,2.193219
30,1.955174
